# *How* del framework anidado de Tamara Munzner en Altair

**Profesor:** Hernán Valdivieso.

In [ ]:
import pandas as pd
import altair as alt
from vega_datasets import data
from vega_datasets import data as vega_data
import IPython

In [ ]:
import requests

def download_file_without_authenticate(id, destination):
    def get_confirm_token(response):
        for key, value in response.cookies.items():
            if key.startswith("download_warning"):
                return value

    URL = "https://docs.google.com/uc?export=download"
    response = requests.get(URL, params={"id": id, "confirm": 1}, stream=True)

    CHUNK_SIZE = 32768
    with open(destination, "wb") as f:
        for i, chunk in enumerate(response.iter_content(CHUNK_SIZE)):
            if chunk:  # filter out keep-alive new chunks
                f.write(chunk)

    return None


download_file_without_authenticate("16HsPD4aCZaiPy9R-bF0KJE6ZyZ6J-cyH", "extintas.csv")

Los datos a utilizar en este notebook corresponden a la información de diferentes lenguajes en peligro de extensión o extintas.

In [ ]:
data = pd.read_csv("extintas.csv")
data.head(6)

# Visual encodings (visualizaciones estáticas)

## Arrange

### Express

Codificamos dos variables cuantitativas usando conjuntamente los canales de posición horizontal y vertical.

In [ ]:
chart = alt.Chart(data).mark_point().encode(
    x="Speakers",
    y="CountriesCount",
)

chart.save('mi_grafico.html', inline=True)
IPython.display.HTML(filename="mi_grafico.html")

In [ ]:
filtrado = data[data.Speakers != 0]

chart_o1 = alt.Chart(filtrado).mark_point().encode(
    x=alt.X("Speakers").scale(type="log"),
    y="CountriesCount",
).properties(width=800, height=500)

chart_o1.save('chart_o1.html', inline=True)
IPython.display.HTML(filename="chart_o1.html")

### Separate
Un mapa de calor es un ejemplo de cómo se pueden separar los datos en 2 ejes. Por ejemplo, en la visualización siguiente, separamos los datos en una matriz de dos dimensiones "grado de peligro" y "número de paises en donde se habla/hablaba dicha lengua", para que así cada celda represente el total de hablantes de dicha lengua según su grado de peligro y el número de paises.

In [ ]:
chart_vb = alt.Chart(data, width=500, height=300).mark_rect().encode(
    y=alt.Y('Endangerment'),
    x=alt.X('CountriesCount:N'),
    color="total_spekears:Q"
).transform_aggregate(
    total_spekears='sum(Speakers)',
    groupby=['Endangerment', 'CountriesCount']
)

chart_vb.save('chart_vb.html', inline=True)
IPython.display.HTML(filename="chart_vb.html")

### Order

Primero vamos a cargar los datos y definir nuesstro propio orden en los datos

In [ ]:
vulnerable_data = data[data.Endangerment == "Vulnerable"]
vulnerable_data = vulnerable_data.sort_values("Speakers", ascending=False)
vulnerable_data = vulnerable_data.head(10) # Notar cómo están ordenadas (primero South Italian	y al final Aymara)
vulnerable_data

Ahora ordenar el eje Y de forma alfabética (comportamiento por defecto en Altair)

In [ ]:
chart_f9 = alt.Chart(vulnerable_data).mark_bar().encode(
    y="Name",
    x="Speakers",
)

chart_f9.save('chart_f9.html', inline=True)
IPython.display.HTML(filename="chart_f9.html")

Otra opción es que nosotros tomemos la decisión de ordenar bajo otro parámetro, por ejemplo, según el valor del eje X (número de hablantes, en nuestro caso)

In [ ]:
chart_o3 = alt.Chart(vulnerable_data).mark_bar().encode(
    y=alt.Y("Name", sort="x"),
    x="Speakers",
)

chart_o3.save('chart_o3.html', inline=True)
IPython.display.HTML(filename="chart_o3.html")

Agregando un "-" podemos invertir el orden

In [ ]:
chart_od = alt.Chart(vulnerable_data).mark_bar().encode(
    y=alt.Y("Name", sort="-x"),
    x="Speakers",
)

chart_od.save('chart_od.html', inline=True)
IPython.display.HTML(filename="chart_od.html")

### Use

Podemos **usar** su información geométrica para hacer un mapa. En este caso, necesitamos primero dibujar el mapa y luego poner los puntos encima.

In [ ]:
countries = alt.topo_feature(vega_data.world_110m.url, 'countries')

In [ ]:
countries

In [ ]:
countries = alt.topo_feature(vega_data.world_110m.url, 'countries')

background = alt.Chart(countries).mark_geoshape(
    fill='lightgray', stroke='white'
)

points = alt.Chart(data).mark_circle().encode(
    longitude='Longitude:Q',
    latitude='Latitude:Q',
    color="Endangerment"
)

chart_emr = background + points
chart_emr.save('chart_emr.html', inline=True)
IPython.display.HTML(filename="chart_emr.html")

# Interactions


## Manipulate (visualizaciones interactivas)

### Change

Una forma de cambio en el tiempo es alternar la visualización. Por ejemplo cambiar su opacidad o color.

In [ ]:
slider = alt.binding_range(min=0, max=1, step=0.05, name='Opacity: ')
op_var = alt.param(value=0.7, bind=slider)

chart_fb = alt.Chart(vulnerable_data).mark_bar(opacity=op_var).encode(
    y=alt.Y("Name"),
    x="Speakers"
).add_params(op_var)

chart_fb.save('chart_fb.html', inline=True)
IPython.display.HTML(filename="chart_fb.html")

In [ ]:
selector = alt.binding_select(options=['red', 'green', 'orange'], name='Color: ')
color_param = alt.param(value="red", bind=selector)

chart_w6 = alt.Chart(vulnerable_data).mark_bar(color=color_param).encode(
    y=alt.Y("Name"),
    x="Speakers"
).add_params(color_param)

chart_w6.save('chart_w6.html', inline=True)
IPython.display.HTML(filename="chart_w6.html")

Tambien puede ser para alterar la visualización en si, como su tamaño y los datos a observar:

In [ ]:
range_ancho = alt.binding_range(min=100, max=1500, name='Ancho: ')
range_speakers = alt.binding_range(min=0, max=8000000, name='Speakers: ')

param_width = alt.param('width', bind=range_ancho, value=300)
param_speakers = alt.param('Speakers', bind=range_speakers, value=0)


chart_oj = alt.Chart(data).mark_circle().encode(
    x="Speakers",
    y="CountriesCount",
    color="Endangerment"
).transform_filter(param_speakers < alt.datum["Speakers"]
).add_params(param_width, param_speakers)

# Como param_width usa el nombre "width", este modifica directamente el ancho de la visualización
chart_oj.save('chart_oj.html', inline=True)
IPython.display.HTML(filename="chart_oj.html")

O incluso hacer cosas más grandes como tener un dividor que nos permita intercalar entre 2 visualizaciones.

In [ ]:
countries = alt.topo_feature(vega_data.world_110m.url, 'countries')

background = alt.Chart(countries).mark_geoshape(
    fill='lightgray', stroke='white'
)


slider = alt.binding_range(min=-180, max=180, step=1)
threshold = alt.param(name="longitud", value=0, bind=slider)

# Gráfico derecha, mostrar solo las lenguas habladas en 1 país
grafico_derecha = alt.Chart(data).mark_circle().encode(
    longitude='Longitude:Q',
    latitude='Latitude:Q',
    color="Endangerment"
).transform_filter(
    (alt.datum["Longitude"] > threshold) & (alt.datum["CountriesCount"] == 1)
)

# Gráfico izquierda, mostrar solo las lenguas Vulnerable habladas en 2 o más países
vulnerables = data[data.Endangerment=="Vulnerable"]
grafico_izquierda = alt.Chart(vulnerables).mark_circle().encode(
    longitude='Longitude:Q',
    latitude='Latitude:Q',
    color="Endangerment"
).transform_filter(
    (alt.datum["Longitude"] < threshold) & (alt.datum["CountriesCount"] != 1)
)

# Nunca olvidar poner add_params para vincular un parámetro a la vis
# El + es para superponer visualizaciones
chart_di = (background + grafico_derecha + grafico_izquierda).add_params(threshold)
chart_di.save('chart_di.html', inline=True)
IPython.display.HTML(filename="chart_di.html")

### Select

#### Select con click

Podemos seleccionar una de las lenguas haciendo click en el punto que lo representa.

In [ ]:
single = alt.selection_point()

chart_1ry = alt.Chart(data).mark_circle(size=100).encode(
    x="Speakers",
    y="CountriesCount",
    color="Endangerment",
    opacity=alt.condition(single, alt.value(1), alt.value(0.01))
).add_params(single)

chart_1ry.save('chart_1ry.html', inline=True)
IPython.display.HTML(filename="chart_1ry.html")

#### Select con hover

Similarmente, podemos seleccionar una de las lenguas con _hover_.

In [ ]:
single_hover = alt.selection_point(on="mouseover", nearest=True)

chart_vc9 = alt.Chart(data).mark_circle(size=100).encode(
    x="Speakers",
    y="CountriesCount",
    color=alt.condition(single_hover, "Endangerment", alt.value("lightgray")),
    order=alt.condition(single_hover, alt.value(1), alt.value(0))
).add_params(
    single_hover
)

chart_vc9.save('chart_vc9.html', inline=True)
IPython.display.HTML(filename="chart_vc9.html")

## Navigate

### Zoom con Altair

Ahora vamos a agregar zoom

In [ ]:
chart_ey = alt.Chart(data).mark_circle().encode(
    x="Speakers",
    y="CountriesCount",
    color="Endangerment",
).interactive() # Una línea hace la magia!!

chart_ey.save('chart_ey.html', inline=True)
IPython.display.HTML(filename="chart_ey.html")

## Facet

### Juxtapose: Vistas coordinadas con Altair

Ahora, intentemos coordinar vistas.

In [ ]:
# Estas dos variables nos permiten definir si queremos los labels de los valores de los ejes y sus títulos.
tick_axis = alt.Axis() # Queremos ver todo
tick_axis_notitle = alt.Axis(title="") # No queremos el título

# Creamos un gráfico de dispersión donde un eje será el atributo "Speakers" y otro eje será "CountriesCount"
points = alt.Chart(data).mark_circle().encode(
    x=alt.X("Speakers", axis=tick_axis_notitle),
    y=alt.Y("CountriesCount", axis=tick_axis_notitle),
    color="Endangerment",
    )

# Para el gráfico que va en el eje X no queremos mostrar el título en el eje y.
x_ticks = alt.Chart(data).mark_tick().encode(
    alt.X("Speakers", axis=tick_axis),
    alt.Y("Endangerment", axis=tick_axis_notitle),
    color="Endangerment"
)

# Para el gráfico que va en el eje Y no queremos mostrar el título en el eje X.
y_ticks = alt.Chart(data).mark_tick().encode(
    alt.X("Endangerment", axis=tick_axis_notitle),
    alt.Y("CountriesCount", axis=tick_axis),
    color="Endangerment"
)

# Se disponen los gráficos en columnas y filas. Las barras verticales | definen las columnas y los & definen filas.
# Una columna será "y_ticks" y la otra columna tendrá 2 filas;
# La primera fila tendrá "points" y la segunda tendrá "x_ticks".
chart_s13 = y_ticks | (points & x_ticks)
chart_s13.save('chart_s13.html', inline=True)
IPython.display.HTML(filename="chart_s13.html")

Ahora, qué tal si agregamos un selector de los datos para enfatizarlos cuando se incluyen en la selección

In [ ]:
# Creamos un objeto de selección que permite seleccionar un intervalo de datos
brush = alt.selection_interval()   ### NUEVA LINEA

tick_axis = alt.Axis()
tick_axis_notitle = alt.Axis(title="")

# El color estará condicionado al objeto selección.
# En caso de seleccionar el dato, usará la columna "Endangerment" para
# definir su color. En otro caso será gris.
# Finalmente agregamos el selector (brush) para que uno pueda seleccionar datos en su gráfico.
points = alt.Chart(data).mark_point().encode(
    x=alt.X("Speakers", axis=tick_axis_notitle),
    y=alt.Y("CountriesCount", axis=tick_axis_notitle),
    color=alt.condition(brush, "Endangerment", alt.value("grey")),  ### NUEVA LINEA
    ).add_params(brush)   ### NUEVA LINEA

x_ticks = alt.Chart(data).mark_tick().encode(
    alt.X("Speakers", axis=tick_axis),
    alt.Y("Endangerment", axis=tick_axis_notitle),
    color=alt.condition(brush, "Endangerment", alt.value("lightgrey")),   ### NUEVA LINEA
    order=alt.condition(brush, alt.value(1), alt.value(0))   ### NUEVA LINEA
)

y_ticks = alt.Chart(data).mark_tick().encode(
    alt.X("Endangerment", axis=tick_axis_notitle),
    alt.Y("CountriesCount", axis=tick_axis),
    color=alt.condition(brush, "Endangerment", alt.value("lightgrey"))   ### NUEVA LINEA
)

chart_zc = y_ticks | (points & x_ticks)
chart_zc.save('chart_zc.html', inline=True)
IPython.display.HTML(filename="chart_zc.html")

Pero el selector no solo puede alterar el color, tambien puede ser una especie de filtro.

In [ ]:
countries = alt.topo_feature(vega_data.world_110m.url, 'countries')
background = alt.Chart(countries).mark_geoshape(
    fill='lightgray', stroke='white'
)

# Creamos un objeto de selección que permite seleccionar un intervalo de datos
brush = alt.selection_interval()

points = alt.Chart(data).mark_point().encode(
    x=alt.X("Speakers"),
    y=alt.Y("CountriesCount"),
    color=alt.condition(brush, "Endangerment", alt.value("grey")),
    ).add_params(brush)

map_1 = alt.Chart(data).mark_circle().encode(
    longitude='Longitude:Q',
    latitude='Latitude:Q',
    color="Endangerment",
).transform_filter(brush)

# Este gráfico no será afectado por el brush
map_2 = alt.Chart(data).mark_circle().encode(
    longitude='Longitude:Q',
    latitude='Latitude:Q',
    color="Endangerment",
)


# Es una fila con 3 columna.
chart_myc = points | (background + map_1) | (background + map_2)
chart_myc.save('chart_myc.html', inline=True)
IPython.display.HTML(filename="chart_myc.html")

Para terminar esta sección, tambien podemos generar nuestros small multiples facilmente con solo 1 parámetro: `facet`.

In [ ]:
chart_w0 = alt.Chart(data).mark_point().encode(
    x=alt.X("Speakers"),
    y=alt.Y("CountriesCount"),
    color="Endangerment:N",
).facet(
    column="Endangerment:N"
)

chart_w0.save('chart_w0.html', inline=True)
IPython.display.HTML(filename="chart_w0.html")

## Reduce

### Filter

#### Filtrar con la leyenda


Altair nos permite agregar interactividad a la leyenda de los gráficos, lo que facilita el filtrado de los datos. A continuación, utilizaremos la leyenda para mostrar solo los datos cuyo origen coincida con el que *clickeemos* en la leyenda.


In [ ]:
                                # Campo sobre el que aplicaremos el filtro
selection = alt.selection_point(fields=["Endangerment"], bind="legend")

chart_fw9 = alt.Chart(data).mark_point().encode(
    x="Speakers",
    y="CountriesCount",
    color="Endangerment",
    # Si el dato coincide con el clickeado, la opacidad es 1. En otro caso 0 (así no se muestra el dato)
    opacity=alt.condition(selection, alt.value(1), alt.value(0))
).add_params(selection)

chart_fw9.save('chart_fw9.html', inline=True)
IPython.display.HTML(filename="chart_fw9.html")

#### Dropdown con Altair

Para seleccionar datos de forma sencilla en nuestras visualizaciones, Altair provee la funcionalidad de _dropdown_. A continuación, se puede ver un ejemplo aplicado al dataset que estamos trabajando en este notebook. Con el dropdown en cuestión, podemos filtrar los datos según el lugar al que pertenecen.

In [ ]:
# Declaramos el dropdown y sus opciones
opciones = ['Vulnerable', 'Definitely endangered', 'Severely endangered',
            'Critically endangered', 'Extinct']

input_dropdown = alt.binding_select(options=opciones, name="Endangerment")

# Creamos una selección enlazada a nuestro dropdown
selection = alt.selection_point(value="Vulnerable", fields=["Endangerment"], bind=input_dropdown)

chart_mde = alt.Chart(data).mark_point().encode(
    y="CountriesCount:Q",
    x="Speakers:Q",
).add_params(selection).transform_filter(selection)

chart_mde.save('chart_mde.html', inline=True)
IPython.display.HTML(filename="chart_mde.html")

Al interactuar con el ejemplo anterior, podemos ver que no es posible volver a mostrar todos los datos una vez que cambiamos el valor seleccionado en el _dropdown_. Para arreglar esto, agregamos la opción None a los posibles valores del _dropdown_.

In [ ]:
# Declaramos el dropdown y sus opciones
opciones = [None, 'Vulnerable', 'Definitely endangered',
            'Severely endangered', 'Critically endangered', 'Extinct']

etiquetas = ["Todo", 'Vulnerable', 'Definitivamente en peligro de extinción',
            'En grave peligro de extinción', 'En peligro crítico', 'Extinto']

input_dropdown = alt.binding_select(options=opciones,
                                    labels=etiquetas,
                                    name="Endangerment")

# Creamos una selección enlazada a nuestro dropdown
selection = alt.selection_point(fields=["Endangerment"], bind=input_dropdown)

chart_6ud = alt.Chart(data).mark_circle().encode(
    x="CountriesCount:Q",
    y="Speakers:Q",
    color="Endangerment:N",
).add_params(
    selection
).transform_filter(
    selection
)

chart_6ud.save('chart_6ud.html', inline=True)
IPython.display.HTML(filename="chart_6ud.html")

#### Buscador de palabras con Altair

Otra interactividad que podemos incluir es agregar un buscador por texto. A continuación, se puede ver un ejemplo aplicado al dataset que estamos trabajando en este notebook. Con el buscador en cuestión, podemos filtrar los datos según el título. Además, utilizaremos Expresión Regular para hacer una busqueda más sofisticada.

In [ ]:
buscador = alt.binding(input='search', placeholder="Nombre lengua",
                       name='Buscar ')

search_input = alt.param(value='', bind=buscador)

countries = alt.topo_feature(vega_data.world_110m.url, 'countries')
background = alt.Chart(countries).mark_geoshape(
    fill='lightgray', stroke='white'
)

points = alt.Chart(data).mark_circle().encode(
    longitude='Longitude:Q',
    latitude='Latitude:Q',
    color="Endangerment",
    tooltip=["Name"]
).transform_filter(
    alt.expr.test(alt.expr.regexp(search_input, 'i'), alt.datum.Name)
)

chart_qvy = (background + points).add_params(search_input)
chart_qvy.save('chart_qvy.html', inline=True)
IPython.display.HTML(filename="chart_qvy.html")

### Aggregate


Altair también nos permite realizar transformaciones sobre los datos a graficar, sin tener que realizar estas operaciones en librerías como Pandas.

In [ ]:
chart_o0p = alt.Chart(data).mark_bar().encode(
    y="Endangerment:O",
    x="average(Speakers):Q"
)

chart_o0p.save('chart_o0p.html', inline=True)
IPython.display.HTML(filename="chart_o0p.html")

Podemos probar también otras transformaciones, como contar cuántas lenguas hay en el dataset para nivel de peligro:

In [ ]:
chart_1an = alt.Chart(data).mark_bar().encode(
    y="Endangerment:O",
    x="count(Endangerment):Q"
)

chart_1an.save('chart_1an.html', inline=True)
IPython.display.HTML(filename="chart_1an.html")

También podemos obtener los mínimos y máximos de los valores de paises que hablan la lengua por nivel de peligro, y compararlos en un gráfico.

In [ ]:
chart_max = alt.Chart(data).mark_bar().encode(
    y="Endangerment:O",
    x="max(CountriesCount):Q",
    color=alt.value("red"),
)

chart_min = alt.Chart(data).mark_bar().encode(
    y="Endangerment:O",
    x="min(CountriesCount):Q",
    color=alt.value("blue"),
)

chart_fd = chart_max + chart_min
chart_fd.save('chart_fd.html', inline=True)
IPython.display.HTML(filename="chart_fd.html")

### Embed

#### Tooltip con Altair

Vamos a agregar _tooltip_ para tener mas info de cada ítem.

In [ ]:
chart_fc = alt.Chart(data).mark_point().encode(
    x="Speakers",
    y="CountriesCount",
    color="Endangerment",
    tooltip=["Name", "CountriesCount", "Speakers"]  # Una línea hace la magia!!
).interactive()

chart_fc.save('chart_fc.html', inline=True)
IPython.display.HTML(filename="chart_fc.html")

# Guardado de visualizaciones

Finalmente, las interacciones que se generan no solo quedan en el notebook. Puedes exportar la visualización interactiva en un HTML y todas las funcionalidades se mantienen, aunque se necesita estar conectado a internet.

A continuación haremos un ejemplo donde aplicaremos yuxtaposición, selección en la leyenda, selección de múltiples datos para filtrar y embed (_tooltip_ en el mapa).

In [ ]:
countries = alt.topo_feature(vega_data.world_110m.url, 'countries')
background = alt.Chart(countries).mark_geoshape(
    fill='lightgray', stroke='white'
)

# Creamos un objeto de selección que permite seleccionar un intervalo de datos
brush = alt.selection_interval()

# Creaomos un objeto selección para permitir seleccionar la leyenda
selection = alt.selection_point(fields=["Endangerment"], bind="legend")


points = alt.Chart(data).mark_point().encode(
    x=alt.X("Speakers"),
    y=alt.Y("CountriesCount"),
    color=alt.condition(brush, "Endangerment", alt.value("grey")),
    opacity=alt.condition(selection, alt.value(1), alt.value(0))
    ).add_params(brush)

points_map = alt.Chart(data).mark_circle().encode(
    longitude='Longitude:Q',
    latitude='Latitude:Q',
    color="Endangerment",
    tooltip=["Name", "CountriesCount", "Speakers"],
    opacity=alt.condition(selection, alt.value(1), alt.value(0))
).transform_filter(brush)


bars = alt.Chart(data).mark_bar().encode(
    x='count()',
    y='Endangerment:N',
    color='Endangerment:N',
    opacity=alt.condition(selection, alt.value(1), alt.value(0.1))
).transform_filter(brush)

yuxtaposicion = ((points & bars) | (background + points_map)).add_params(selection)
yuxtaposicion.save("chart.html", inline=True)

In [ ]:
import IPython
IPython.display.HTML(filename="chart.html")

# Fuentes utilizadas en este notebook

* Documentacion de Altair: https://altair-viz.github.io/gallery/index.html#interactive-charts
* Interacciones con Altair: https://colab.research.google.com/github/uwdata/visualization-curriculum/blob/master/altair_interaction.ipynb#scrollTo=NWVWj-hYQqcx
* Tutorial adicional: https://medium.com/analytics-vidhya/interactive-data-viz-using-altair-873139771fe2